# COVID-19 Global Impact Dashboard
**Goal:** clean time-series data, derive per-million metrics, rolling averages and vaccination indicators, then build a portfolio dashboard.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT=Path('..').resolve()
DATA=ROOT/'data'/'processed'/'owid_covid_cleaned.csv'
df=pd.read_csv(DATA, parse_dates=['date'])
df.head()

## 1. Data quality checks

In [ ]:
print('Shape:', df.shape)
print('Date range:', df.date.min(), df.date.max())
print('Countries:', df.location.nunique())
print('Missing values (top 15):')
print(df.isna().sum().sort_values(ascending=False).head(15))

## 2. Rolling averages and country comparison

In [ ]:
country='India'
sub=df[df.location.eq(country)].sort_values('date')
fig,ax=plt.subplots(figsize=(12,5))
ax.plot(sub.date,sub.cases_7d_avg_per_million,label='Cases 7-day avg / million')
ax.plot(sub.date,sub.deaths_7d_avg_per_million,label='Deaths 7-day avg / million')
ax.set_title(f'{country} COVID-19 rolling trend')
ax.legend(); ax.grid(alpha=.2); plt.show()

## 3. Country summary

In [ ]:
latest=df.sort_values('date').groupby('location').tail(1)
summary=latest[['location','population','total_cases_per_million','total_deaths_per_million','people_vaccinated_per_hundred','case_fatality_rate_pct']].sort_values('total_cases_per_million',ascending=False)
summary.head(15)

## 4. Vaccination vs deaths

In [ ]:
fig,ax=plt.subplots(figsize=(9,6))
ax.scatter(latest.people_vaccinated_per_hundred, latest.total_deaths_per_million)
for _,r in latest.iterrows():
    ax.annotate(r.location,(r.people_vaccinated_per_hundred,r.total_deaths_per_million),fontsize=7,xytext=(3,3),textcoords='offset points')
ax.set_xlabel('People vaccinated (%)'); ax.set_ylabel('Deaths per million'); ax.set_title('Vaccination coverage vs cumulative deaths')
ax.grid(alpha=.2); plt.show()

## 5. Plain-language interpretation
- Use per-million metrics for fairer country comparisons.
- Use 7-day rolling averages to reduce daily reporting noise.
- Compare vaccination coverage with mortality descriptively, not as proof of causality.
- Investigate outliers separately because reporting practices, age structure, testing and healthcare capacity differ.